In [4]:
import numpy as np
from ortools.linear_solver import pywraplp
from warnings import filterwarnings

filterwarnings("ignore")

# Reading the cost matrix and the pair matrix
cost_matrix = np.loadtxt("../data/cost_matrix/cost.txt").reshape(-1, 1)
pair_matrix = np.genfromtxt("../data/pairings/pair_array.txt", delimiter=',', dtype='int')

# Determining the number of flights and tasks
num_pairs = pair_matrix.shape[0]
num_flights = pair_matrix.shape[1]

# Initialize MIP Solver
solver = pywraplp.Solver.CreateSolver("SCIP")

# Binary variables for existing columns
x = np.array([solver.BoolVar("") for _ in range(num_pairs)]).reshape(-1, 1)

# Declare constraints for existing columns
for i in range(num_flights):
    solver.Add(solver.Sum(x[j][0] * pair_matrix[j][i] for j in range(num_pairs)) == 1)

# Declare objective function for existing columns
solver.Minimize(np.sum(cost_matrix * x))

# Solve the problem
solver.Solve()

# Initialize column generation loop
while True:
    reduced_costs = cost_matrix - np.dot(np.transpose(x), pair_matrix).reshape(-1, 1)

    # Create new column (variable) with negative reduced cost
    y = solver.BoolVar("")
    solver.Add(solver.Sum(y * reduced_costs) <= 0)

    # Solve the problem with the new column
    solver.Solve()

    # Check for optimality
    if reduced_costs.max() >= 0:
        break

    # Add the new column to the constraints
    new_column = np.array([y.solution_value()])
    x = np.vstack((x, new_column))

# Extract the selected crew pairings
selected_pairings = [i for i in range(num_pairs) if x[i][0].solution_value() > 0.5]

# Output the selected pairings
with open(f"../data/selected_pairings/selected_pairings.txt", "w") as file:
    for idx in selected_pairings:
        file.write(f"{pair_matrix[idx]}\n")

ValueError: operands could not be broadcast together with shapes (776,1) (60,1) 

In [3]:
import numpy as np
from ortools.linear_solver import pywraplp
from warnings import filterwarnings

filterwarnings("ignore")

# Reading the cost matrix and the pair matrix
cost_matrix = np.loadtxt("../data/cost_matrix/cost.txt").reshape(-1, 1)
pair_matrix = np.genfromtxt("../data/pairings/pair_array.txt", delimiter=',', dtype='int')

# Determining the number of flights and tasks
num_pairs = pair_matrix.shape[0]
num_flights = pair_matrix.shape[1]

# Initialize MIP Solver
solver = pywraplp.Solver.CreateSolver("SCIP")

# Creating the binary allocation variable
x = np.array([solver.BoolVar("") for _ in range(num_pairs)]).reshape(-1, 1)
result_matrix = x * pair_matrix

# Declare constraints
for i in range(num_flights):
    solver.Add(solver.Sum(result_matrix[:, i]) == 1)

# Declare objective function
solver.Minimize(np.sum(cost_matrix * x))

# Solve the problem
status = solver.Solve()

# Print the state of the optimization problem
if status == pywraplp.Solver.OPTIMAL:
    print("The Solution is OPTIMAL")
elif status == pywraplp.Solver.FEASIBLE:
    print("The Solution is Feasible")
else:
    print("The Solution is not Feasible")

# Extract the selected crew pairings
selected_pairings = [i for i in range(num_pairs) if x[i][0].solution_value() > 0.5]
print(selected_pairings)
print(solver.Objective().Value())

# Output the selected pairings
with open(f"../data/selected_pairings/selected_pairings.txt", "w") as file:
    for idx in selected_pairings:
        file.write(f"{pair_matrix[idx]}\n")


The Solution is OPTIMAL
[775]
20.0


FileNotFoundError: [Errno 2] No such file or directory: '../data/selected_pairings/selected_pairings.txt'